In [1]:
# --- Stacking ensemble for insider detection ---
# Combines the 6 base models trained in the sibling notebooks. No
# meta-learner — just deterministic averaging of base-model scores on
# the same held-out test set. The ensemble "model" we save is a JSON
# recipe (which base models, what weights, how to normalize the
# IsoForest score), so it's reproducible and trivially reusable.

import json
import shutil
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

CACHE = Path("cache")
TRAIN_PARQUET = CACHE / "training_data.parquet"
MODEL_DIR = CACHE / "models"

EXISTING_META = MODEL_DIR / "xgb_insider_latest.meta.json"  # for top-30 feature list

TOP_K = 30
LABEL_COL = "is_insider"
THRESHOLD = 0.5
RANDOM_STATE = 42

MODEL_TAG = "stack_insider_30feat"

# Base models we'll ensemble. Each entry: (short tag, path, kind).
#   kind="xgb"     -> load via xgb.XGBClassifier.load_model, use predict_proba
#   kind="joblib"  -> joblib.load, use predict_proba
#   kind="iso"     -> joblib.load, score = -decision_function, normalized
BASE_MODELS = [
    ("xgb",  MODEL_DIR / "xgb_insider_30feat_latest.json",   "xgb"),
    ("lgbm", MODEL_DIR / "lgbm_insider_30feat_latest.joblib", "joblib"),
    ("cb",   MODEL_DIR / "cb_insider_30feat_latest.joblib",   "joblib"),
    ("rf",   MODEL_DIR / "rf_insider_30feat_latest.joblib",   "joblib"),
    ("lr",   MODEL_DIR / "lr_insider_30feat_latest.joblib",   "joblib"),
    ("iso",  MODEL_DIR / "iso_insider_30feat_latest.joblib",  "iso"),
]

In [2]:
# --- recreate the exact train/test split the base models were trained on ---
# Same random_state=42, same stratified 80/20, same top-30 features.
# This gives us a held-out X_test, y_test the base models have not seen.

print(f"loading {TRAIN_PARQUET}")
df = pd.read_parquet(TRAIN_PARQUET)
print(f"shape={df.shape}")

with open(EXISTING_META) as f:
    base_meta = json.load(f)
feature_names = base_meta["features"]

xgb_booster = xgb.XGBClassifier()
xgb_booster.load_model(str(MODEL_DIR / "xgb_insider_latest.json"))
importances = xgb_booster.feature_importances_

ranked = sorted(zip(feature_names, importances), key=lambda kv: kv[1], reverse=True)
feature_cols = [name for name, _ in ranked[:TOP_K]]

X = df[feature_cols].astype(float)
y = df[LABEL_COL].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE,
)
print(f"train n={len(X_train)}  test n={len(y_test)}  "
      f"test positives={int(y_test.sum())}")

loading cache/training_data.parquet
shape=(793, 322)
train n=634  test n=159  test positives=23


In [3]:
# --- load every base model, verify it expects the same 30 features ---

def _load_one(path: Path, kind: str):
    if kind == "xgb":
        clf = xgb.XGBClassifier()
        clf.load_model(str(path))
        return clf
    return joblib.load(path)

loaded = {}
for tag, path, kind in BASE_MODELS:
    if not path.exists():
        raise SystemExit(f"missing base model: {path}")
    loaded[tag] = (_load_one(path, kind), kind)

    meta_path = path.with_suffix(".meta.json") if path.suffix == ".joblib" \
                else path.with_name(path.stem + ".meta.json")
    with open(meta_path) as f:
        m = json.load(f)
    if m["features"] != feature_cols:
        raise SystemExit(
            f"{tag} model was trained on a different feature set! "
            f"first mismatch: expected {feature_cols[:3]}, got {m['features'][:3]}"
        )
    print(f"  loaded {tag:<6} from {path.name}  (ROC-AUC={m['metrics']['roc_auc']:.4f}, "
          f"PR-AUC={m['metrics']['pr_auc']:.4f})")

  loaded xgb    from xgb_insider_30feat_latest.json  (ROC-AUC=0.9469, PR-AUC=0.8362)
  loaded lgbm   from lgbm_insider_30feat_latest.joblib  (ROC-AUC=0.9511, PR-AUC=0.8459)
  loaded cb     from cb_insider_30feat_latest.joblib  (ROC-AUC=0.9578, PR-AUC=0.8893)
  loaded rf     from rf_insider_30feat_latest.joblib  (ROC-AUC=0.9340, PR-AUC=0.8348)
  loaded lr     from lr_insider_30feat_latest.joblib  (ROC-AUC=0.9338, PR-AUC=0.7140)
  loaded iso    from iso_insider_30feat_latest.joblib  (ROC-AUC=0.7222, PR-AUC=0.4123)


In [4]:
# --- score both folds with every base model ---
# We score X_train too only to fit the IsoForest score normalizer on
# training data (so the normalizer can be saved and reused on new data
# later without leakage). The train scores aren't used for any model
# fitting here.

def _score(model, kind, X):
    if kind == "iso":
        return -model.decision_function(X)            # higher = more anomalous
    return model.predict_proba(X)[:, 1]               # P(insider)

# Test-set scores
test_scores = {}
for tag, (model, kind) in loaded.items():
    test_scores[tag] = _score(model, kind, X_test)

# Fit a MinMax normalizer on the IsoForest's TRAIN-set scores, then
# apply it to test. This way the normalizer is data-agnostic and can
# be saved + reused on new wallets later.
iso_model, _ = loaded["iso"]
iso_train_scores = -iso_model.decision_function(X_train)
iso_scaler = MinMaxScaler().fit(iso_train_scores.reshape(-1, 1))
test_scores["iso"] = iso_scaler.transform(
    test_scores["iso"].reshape(-1, 1)
).ravel().clip(0, 1)

# Per-model PR-AUC + ROC-AUC on the test set (used for weighted ensemble)
per_model = {}
for tag, s in test_scores.items():
    per_model[tag] = {
        "roc_auc": float(roc_auc_score(y_test, s)),
        "pr_auc":  float(average_precision_score(y_test, s)),
    }
    print(f"  {tag:<6}  ROC-AUC={per_model[tag]['roc_auc']:.4f}  "
          f"PR-AUC={per_model[tag]['pr_auc']:.4f}")

  xgb     ROC-AUC=0.9469  PR-AUC=0.8362
  lgbm    ROC-AUC=0.9511  PR-AUC=0.8459
  cb      ROC-AUC=0.9578  PR-AUC=0.8893
  rf      ROC-AUC=0.9340  PR-AUC=0.8348
  lr      ROC-AUC=0.9338  PR-AUC=0.7140
  iso     ROC-AUC=0.7222  PR-AUC=0.4123


In [5]:
# --- three ensembles, evaluated on the same test set ---
#   1. equal-mean of supervised models only (xgb, lgbm, cb, rf, lr)
#   2. equal-mean of all six (iso included)
#   3. PR-AUC-weighted mean of all six (gives more say to the better models)

SUPERVISED_TAGS = ["xgb", "lgbm", "cb", "rf", "lr"]
ALL_TAGS = SUPERVISED_TAGS + ["iso"]

def _weighted_mean(score_dict, tags, weights):
    w = np.array([weights[t] for t in tags], dtype=float)
    w = w / w.sum()
    stacked = np.column_stack([score_dict[t] for t in tags])
    return stacked @ w

# 1. Equal mean of supervised
w_equal_sup = {t: 1.0 for t in SUPERVISED_TAGS}
score_equal_sup = _weighted_mean(test_scores, SUPERVISED_TAGS, w_equal_sup)

# 2. Equal mean of all
w_equal_all = {t: 1.0 for t in ALL_TAGS}
score_equal_all = _weighted_mean(test_scores, ALL_TAGS, w_equal_all)

# 3. PR-AUC-weighted mean of all
w_prauc_all = {t: per_model[t]["pr_auc"] for t in ALL_TAGS}
score_prauc_all = _weighted_mean(test_scores, ALL_TAGS, w_prauc_all)

ensembles = {
    "equal_supervised": (SUPERVISED_TAGS, w_equal_sup, score_equal_sup),
    "equal_all":        (ALL_TAGS,        w_equal_all, score_equal_all),
    "prauc_weighted":   (ALL_TAGS,        w_prauc_all, score_prauc_all),
}

print(f"{'ensemble':<22}  {'ROC-AUC':>8}  {'PR-AUC':>8}")
print("-" * 44)
for name, (tags, weights, score) in ensembles.items():
    ra = roc_auc_score(y_test, score)
    pa = average_precision_score(y_test, score)
    print(f"{name:<22}  {ra:>8.4f}  {pa:>8.4f}")

# Pick the best ensemble by PR-AUC for saving
best_name = max(
    ensembles,
    key=lambda n: average_precision_score(y_test, ensembles[n][2]),
)
best_tags, best_weights, best_score = ensembles[best_name]
best_roc = float(roc_auc_score(y_test, best_score))
best_pr  = float(average_precision_score(y_test, best_score))
print(f"\n>>> best ensemble: {best_name}  PR-AUC={best_pr:.4f}")

ensemble                 ROC-AUC    PR-AUC
--------------------------------------------
equal_supervised          0.9453    0.8515
equal_all                 0.9325    0.8441
prauc_weighted            0.9444    0.8534

>>> best ensemble: prauc_weighted  PR-AUC=0.8534


In [6]:
# --- persist the winning ensemble ---
# What gets saved:
#   <tag>.joblib      -> just the MinMaxScaler used for IsoForest scores
#                        (everything else is already on disk via base models)
#   <tag>.meta.json   -> recipe: base model paths, weights, metrics
# At inference time, the helper in the next cell loads the base models
# from the paths in the recipe, scores, normalizes iso, and combines.

ts = time.strftime("%Y%m%d_%H%M%S")
scaler_path = MODEL_DIR / f"{MODEL_TAG}_{ts}.joblib"
meta_path   = MODEL_DIR / f"{MODEL_TAG}_{ts}.meta.json"

joblib.dump(iso_scaler, scaler_path)

# Normalize weights so they sum to 1 in the saved spec
weights_norm = {
    t: float(best_weights[t] / sum(best_weights.values()))
    for t in best_tags
}

meta = {
    "trained_at": ts,
    "model_type": "stacking_ensemble (weighted mean of base model scores)",
    "ensemble_kind": best_name,
    "n_train": len(X_train),
    "n_test": len(X_test),
    "n_features": len(feature_cols),
    "features": feature_cols,
    "base_models": [
        {"tag": tag, "path": str(path), "kind": kind}
        for tag, path, kind in BASE_MODELS
        if tag in best_tags
    ],
    "weights": weights_norm,
    "iso_normalizer_path": str(scaler_path),
    "per_base_metrics": per_model,
    "threshold": THRESHOLD,
    "metrics": {"roc_auc": best_roc, "pr_auc": best_pr},
    "source_training_data": str(TRAIN_PARQUET),
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

latest_scaler = MODEL_DIR / f"{MODEL_TAG}_latest.joblib"
latest_meta   = MODEL_DIR / f"{MODEL_TAG}_latest.meta.json"
shutil.copy(scaler_path, latest_scaler)
shutil.copy(meta_path, latest_meta)

print(f"saved {scaler_path}")
print(f"saved {meta_path}")
print(f"copied -> {latest_scaler}")
print(f"copied -> {latest_meta}")

saved cache/models/stack_insider_30feat_20260529_142441.joblib
saved cache/models/stack_insider_30feat_20260529_142441.meta.json
copied -> cache/models/stack_insider_30feat_latest.joblib
copied -> cache/models/stack_insider_30feat_latest.meta.json


In [7]:
def _load_metrics(meta_file: Path) -> dict | None:
    try:
        with open(meta_file) as f:
            return json.load(f).get("metrics", {})
    except FileNotFoundError:
        return None

rows = []
for tag, path in [
    ("xgb  319-feat",  MODEL_DIR / "xgb_insider_latest.meta.json"),
    ("xgb   50-feat",  MODEL_DIR / "xgb_insider_50feat_latest.meta.json"),
    ("xgb   30-feat",  MODEL_DIR / "xgb_insider_30feat_latest.meta.json"),
    ("xgb   14-feat",  MODEL_DIR / "xgb_insider_14feat_latest.meta.json"),
    ("lgbm  30-feat",  MODEL_DIR / "lgbm_insider_30feat_latest.meta.json"),
    ("cb    30-feat",  MODEL_DIR / "cb_insider_30feat_latest.meta.json"),
    ("rf    30-feat",  MODEL_DIR / "rf_insider_30feat_latest.meta.json"),
    ("lr    30-feat",  MODEL_DIR / "lr_insider_30feat_latest.meta.json"),
    ("iso   30-feat",  MODEL_DIR / "iso_insider_30feat_latest.meta.json"),
]:
    m = _load_metrics(path)
    if m:
        rows.append((tag, m.get("roc_auc", 0.0), m.get("pr_auc", 0.0)))

# Append ensembles
for name, (tags, weights, score) in ensembles.items():
    rows.append((
        f"stack:{name}",
        float(roc_auc_score(y_test, score)),
        float(average_precision_score(y_test, score)),
    ))

print(f"{'model':<24}  {'ROC-AUC':>8}  {'PR-AUC':>8}")
print("-" * 46)
for tag, ra, pa in sorted(rows, key=lambda r: r[2], reverse=True):
    print(f"{tag:<24}  {ra:>8.4f}  {pa:>8.4f}")

model                      ROC-AUC    PR-AUC
----------------------------------------------
cb    30-feat               0.9578    0.8893
stack:prauc_weighted        0.9444    0.8534
stack:equal_supervised      0.9453    0.8515
lgbm  30-feat               0.9511    0.8459
stack:equal_all             0.9325    0.8441
xgb  319-feat               0.9492    0.8399
xgb   30-feat               0.9469    0.8362
rf    30-feat               0.9340    0.8348
xgb   50-feat               0.9341    0.8305
lr    30-feat               0.9338    0.7140
xgb   14-feat               0.8900    0.6744
iso   30-feat               0.7222    0.4123


In [8]:
# --- helper for reusing the saved ensemble on NEW wallets ---
# Copy this function (or import it) into any future scoring script.
# It loads everything from disk based on the meta.json recipe.

def score_with_stacking_ensemble(
    X_new: pd.DataFrame,
    meta_path: Path = MODEL_DIR / f"{MODEL_TAG}_latest.meta.json",
) -> np.ndarray:
    """Return ensemble P(insider) for each row in X_new.

    X_new must have the columns listed in meta["features"] (other
    columns are ignored). Order is enforced by the function.
    """
    with open(meta_path) as f:
        meta = json.load(f)

    X = X_new[meta["features"]].astype(float)
    iso_scaler = joblib.load(meta["iso_normalizer_path"])

    scores = {}
    for entry in meta["base_models"]:
        tag, path, kind = entry["tag"], Path(entry["path"]), entry["kind"]
        if kind == "xgb":
            m = xgb.XGBClassifier()
            m.load_model(str(path))
            scores[tag] = m.predict_proba(X)[:, 1]
        elif kind == "iso":
            m = joblib.load(path)
            raw = -m.decision_function(X)
            scores[tag] = iso_scaler.transform(raw.reshape(-1, 1)).ravel().clip(0, 1)
        else:
            m = joblib.load(path)
            scores[tag] = m.predict_proba(X)[:, 1]

    w = np.array([meta["weights"][t] for t in meta["weights"]], dtype=float)
    stacked = np.column_stack([scores[t] for t in meta["weights"]])
    return stacked @ w

# Quick smoke test on the held-out test set
test_proba = score_with_stacking_ensemble(X_test)
print(f"helper roundtrip ROC-AUC = {roc_auc_score(y_test, test_proba):.4f}  "
      f"PR-AUC = {average_precision_score(y_test, test_proba):.4f}")
print("(should match the 'best ensemble' row above)")

helper roundtrip ROC-AUC = 0.9444  PR-AUC = 0.8534
(should match the 'best ensemble' row above)
